# Cat Diffusion — Metrics & Generation Comparison

Compares the two VP-Exponential checkpoints produced by `train_cats.py`:

| Model | Checkpoint |
|-------|------------|
| **Phase 1** | `cat64_VP-Exp_phase1_lr1e-03_ep1500.pth` (lr = 1e-3, from scratch) |
| **Phase 2** | `cat64_VP-Exp_phase2_lr1e-04_ep1500.pth` (lr = 1e-4, fine-tuned) |

Metrics computed on a held-out test split of the cat dataset:
- **FID** — Fréchet Inception Distance (generated vs real)
- **BPD** — Bits per dimension via probability flow ODE
- **IS** — Inception Score (generated images)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import ToTensor

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'diffusion_lib').exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from diffusion_lib import (
    VPProcess, ExponentialSchedule,
    EulerMaruyamaSampler,
    UNetScoreModelColor, GenerativeDiffusionModel,
)
from diffusion_lib.metrics.fid_is import compute_fid, compute_is
from diffusion_lib.metrics.bpd import compute_bpd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
class CatFolderDataset(Dataset):
    def __init__(self, folder_path):
        paths = sorted(Path(folder_path).glob('*.jpg'))
        transform = ToTensor()
        print(f'Loading {len(paths)} images...')
        self.images = [transform(Image.open(p).convert('RGB')) for p in paths]
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx], 0

CAT_FOLDER = PROJECT_DIR / 'cat_AI_v2' / 'cats'
full_dataset = CatFolderDataset(CAT_FOLDER)

# 90/10 split, seed fijo
n_test  = max(50, len(full_dataset) // 10)
n_train = len(full_dataset) - n_test
_, test_set = random_split(full_dataset, [n_train, n_test],
                            generator=torch.Generator().manual_seed(42))

x_test = torch.stack([test_set[i][0] for i in range(len(test_set))])
print(f'Test split: {len(x_test)} images  |  shape: {x_test.shape}')

## Load models

In [ ]:
CKPT_DIR = PROJECT_DIR / 'cat_AI_v2' / 'checkpoints'

CHECKPOINTS = {
    'Phase 1 (lr=1e-3)': CKPT_DIR / 'cat64_VP-Exp_phase1_lr1e-03_ep1500.pth',
    'Phase 2 (lr=1e-4)': CKPT_DIR / 'cat64_VP-Exp_phase2_lr1e-04_ep1500.pth',
}

process = VPProcess(schedule=ExponentialSchedule())
models  = {}

for name, ckpt_path in CHECKPOINTS.items():
    m = UNetScoreModelColor(marginal_prob_std=process.sigma_t).to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    m.eval()
    models[name] = m
    print(f'Loaded [{name}]: {ckpt_path.name}')

## 1 · Visual generation

In [ ]:
N_GEN     = 20
N_STEPS   = 500
IMG_SHAPE = (3, 64, 64)

generated = {}
for name, score_model in models.items():
    print(f'Sampling {N_GEN} images — {name}...', end=' ', flush=True)
    gm   = GenerativeDiffusionModel(process, EulerMaruyamaSampler(), score_model, device)
    imgs = gm.sample(n_images=N_GEN, img_shape=IMG_SHAPE, n_steps=N_STEPS)
    generated[name] = imgs.cpu().clamp(0, 1)
    print('done')

In [ ]:
from torchvision.utils import make_grid

out_dir = PROJECT_DIR / 'Figuras'
out_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 1, figsize=(N_GEN * 1.1, 4.5))
for ax, (name, imgs) in zip(axes, generated.items()):
    grid = make_grid(imgs, nrow=N_GEN, padding=2).permute(1, 2, 0).numpy()
    ax.imshow(grid)
    ax.set_title(name, fontsize=12)
    ax.axis('off')
plt.tight_layout()
fig.savefig(out_dir / 'cats_generation_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## 2 · FID — generated vs real cats

In [ ]:
N_FID = 500

fid_scores = {}
for name, score_model in models.items():
    print(f'FID [{name}]: generating {N_FID} samples...', end=' ', flush=True)
    gm   = GenerativeDiffusionModel(process, EulerMaruyamaSampler(), score_model, device)
    fake = gm.sample(n_images=N_FID, img_shape=IMG_SHAPE, n_steps=N_STEPS).cpu().clamp(0, 1)
    fid  = compute_fid(x_test, fake, device=device)
    fid_scores[name] = fid
    print(f'FID = {fid:.2f}')

print('\nFID summary:')
for name, fid in fid_scores.items():
    print(f'  {name:<25}  {fid:.2f}')

## 3 · BPD — bits per dimension (probability flow ODE)

In [ ]:
N_BPD_STEPS  = 50
N_HUTCHINSON = 1
BPD_BATCH    = 16

bpd_scores = {}
for name, score_model in models.items():
    bpd_list = []
    for start in range(0, len(x_test), BPD_BATCH):
        batch     = x_test[start:start + BPD_BATCH].to(device)
        bpd_batch = compute_bpd(score_model, process, batch,
                                n_steps=N_BPD_STEPS, n_hutchinson=N_HUTCHINSON)
        bpd_list.append(bpd_batch.cpu())
    mean_bpd = torch.cat(bpd_list).mean().item()
    bpd_scores[name] = mean_bpd
    print(f'BPD [{name}]: {mean_bpd:.4f}')

## 4 · Inception Score

In [ ]:
is_scores = {}
for name, imgs in generated.items():
    is_mean, is_std = compute_is(imgs, device=device)
    is_scores[name] = (is_mean, is_std)
    print(f'IS [{name}]: {is_mean:.3f} ± {is_std:.3f}')

## 5 · Summary table + bar chart

In [ ]:
names = list(models.keys())

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# FID
ax = axes[0]
bars = ax.bar(names, [fid_scores[n] for n in names], color=['steelblue', 'darkorange'])
ax.bar_label(bars, fmt='%.1f', padding=3)
ax.set_title('FID ↓ (lower is better)')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=12, ha='right', fontsize=9)

# BPD
ax = axes[1]
bars = ax.bar(names, [bpd_scores[n] for n in names], color=['steelblue', 'darkorange'])
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.set_title('BPD ↓ (lower is better)')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=12, ha='right', fontsize=9)

# IS
ax = axes[2]
means = [is_scores[n][0] for n in names]
stds  = [is_scores[n][1] for n in names]
bars  = ax.bar(names, means, yerr=stds, color=['steelblue', 'darkorange'], capsize=5)
ax.bar_label(bars, fmt='%.2f', padding=3)
ax.set_title('IS ↑ (higher is better)')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=12, ha='right', fontsize=9)

plt.suptitle('Cat Diffusion — Phase 1 vs Phase 2', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(out_dir / 'cats_metrics_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n=== Final summary ===')
print(f'{"Model":<25}  {"FID":>8}  {"BPD":>8}  {"IS":>12}')
print('-' * 60)
for n in names:
    print(f'{n:<25}  {fid_scores[n]:>8.2f}  {bpd_scores[n]:>8.4f}  '
          f'{is_scores[n][0]:>6.3f} ± {is_scores[n][1]:.3f}')